_Imports_

In [ ]:
import os
import cv2
import json
import shutil
import hashlib
import warnings
import subprocess
from tqdm import tqdm
from PIL import Image
from mutagen.mp4 import MP4
from datetime import datetime
from PIL.ExifTags import TAGS

Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore")

_Constants_

In [ ]:
imageExtensions = {'.jpg', '.jpeg', '.png', '.heic', '.webp'}
videoExtensions = {'.mp4', '.avi', '.mkv', '.mov', '.webm'}
allExtensions = imageExtensions.union(videoExtensions)

memoriesFolder = "path\to\your\folder"

_Find Files_

In [23]:
def findFiles(folderPath: str) -> list[str]:
    matchingFiles = []

    for root, _, files in os.walk(folderPath):
        for file in files:
            if os.path.splitext(file)[1].lower() in allExtensions:
                matchingFiles.append(os.path.join(root, file))

    return matchingFiles

foundedMemories = findFiles(memoriesFolder)
print(f"Found {len(foundedMemories)} files in {memoriesFolder}")

Found 12052 files in C:\Users\nagin\OneDrive\Belgeler\Memories


_Convert HEIF Files_

In [24]:
def convertToJPG(inputPath: str):
    outputPath = os.path.splitext(inputPath)[0] + ".jpg"
    subprocess.run(["ffmpeg", "-i", inputPath, outputPath])
    return outputPath

heicMemories = [file for file in foundedMemories if os.path.splitext(file)[1].lower() == ".heic"]
if len(heicMemories) == 0:
    print("No HEIC files found for conversion.")
else:
    for heicMemory in tqdm(heicMemories, desc="Converting HEIC to JPG"):
        outputPath = convertToJPG(heicMemory)
        if os.path.exists(outputPath):
            foundedMemories.append(outputPath)
            foundedMemories.remove(heicMemory)

No HEIC files found for conversion.


_Remove Garbage Files_

In [25]:
def hasUnusualName(filePath: str) -> bool:
    nameWithoutExt = os.path.splitext(os.path.basename(filePath))[0].lower()
    keywords = {"trash", "stk", "tmp", "temp", "cache"}
    return any(kw in nameWithoutExt for kw in keywords)

cleanMemories = []
garbageCount = 0
for memory in tqdm(foundedMemories, desc="Removing garbage files"):
    if hasUnusualName(memory):
        garbageCount += 1
    else:
        cleanMemories.append(memory)
foundedMemories = cleanMemories

if garbageCount == 0:
    print("No garbage files found.")
else:
    print(f"Removed {garbageCount} garbage files.")


Removing garbage files: 100%|██████████| 12052/12052 [00:00<00:00, 37256.82it/s]

Removed 100 garbage files.


_Check Corrupted Files_

In [26]:
def isCorrupted(filePath: str) -> bool:
    ext = os.path.splitext(filePath)[1].lower()
    if ext in imageExtensions:
        try:
            img = Image.open(filePath)
            img.verify()
            return False
        except Exception:
            return True
    return False

validMemories = []
corruptedCount = 0
for memory in tqdm(foundedMemories, desc="Checking for corrupted files"):
    if isCorrupted(memory):
        corruptedCount += 1
    else:
        validMemories.append(memory)
foundedMemories = validMemories

if corruptedCount == 0:
    print("No corrupted files found.")
else:
    print(f"Removed {corruptedCount} corrupted files.")


Checking for corrupted files: 100%|██████████| 11952/11952 [00:24<00:00, 479.12it/s]

No corrupted files found.


_Convert WEBP to JPG_

In [27]:
webpMemories = [file for file in foundedMemories if os.path.splitext(file)[1].lower() == ".webp"]
if len(webpMemories) == 0:
    print("No WebP files found for conversion.")
else:
    for webpMemory in tqdm(webpMemories, desc="Converting WebP to JPG"):
        outputPath = convertToJPG(webpMemory)
        if os.path.exists(outputPath):
            foundedMemories.append(outputPath)
            foundedMemories.remove(webpMemory)

No WebP files found for conversion.


_Convert Videos to MP4_

In [28]:
def convertToMp4(inputPath: str):
    outputPath = os.path.splitext(inputPath)[0] + ".mp4"
    subprocess.run(["ffmpeg", "-i", inputPath, "-c:v", "libx264", "-c:a", "aac", outputPath])
    return outputPath

videoMemories = [file for file in foundedMemories if os.path.splitext(file)[1].lower() in {".avi", ".mkv", ".mov", ".webm"}]

if len(videoMemories) == 0:
    print("No video files found for conversion.")
else:
    for videoMemory in tqdm(videoMemories, desc="Converting video to MP4"):
        outputPath = convertToMp4(videoMemory)
        if os.path.exists(outputPath):
            foundedMemories.append(outputPath)
            foundedMemories.remove(videoMemory)

No video files found for conversion.


_Check File Dimensions_

In [29]:
def getDimensions(filePath: str):
    ext = filePath.rsplit(".", 1)[-1].lower()
    if ext in ("jpg", "jpeg", "png"):
        img = Image.open(filePath)
        return img.size
    elif ext == "mp4":
        cap = cv2.VideoCapture(filePath)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        return w, h
    return None, None

highResMemories = []
countLowResFiles = 0
for memory in tqdm(foundedMemories, desc="Getting dimensions of files"):
    width, height = getDimensions(memory)
    if width is None or height is None:
        highResMemories.append(memory)
    elif height < 720 or width < 720:
        countLowResFiles += 1
    else:
        highResMemories.append(memory)
foundedMemories = highResMemories

if countLowResFiles == 0:
    print("No low resolution files found.")
else:
    print(f"Removed {countLowResFiles} low resolution files.")


Getting dimensions of files: 100%|██████████| 11952/11952 [01:54<00:00, 104.31it/s] 

Removed 1384 low resolution files.


_Remove Duplicates_

In [30]:
def imageHash(path: str):
    img = Image.open(path).convert("RGB").resize((32, 32))
    return hashlib.md5(img.tobytes()).hexdigest()

def videoHash(path: str):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

def removeDuplicates(files):
    seenImages = {}
    seenVideos = {}
    result = []

    for file in tqdm(files, desc="Removing duplicates"):
        ext = file.split(".")[-1].lower()

        if ext in ["jpg", "jpeg", "png"]:
            h = imageHash(file)

            if h in seenImages:
                old = seenImages[h]
                if old.endswith((".jpg", ".jpeg")) and ext == "png":
                    result[result.index(old)] = file
                    seenImages[h] = file
            else:
                seenImages[h] = file
                result.append(file)

        elif ext == "mp4":
            h = videoHash(file)
            if h not in seenVideos:
                seenVideos[h] = file
                result.append(file)

    return result

deduplicatedMemories = removeDuplicates(foundedMemories)
print(f"Removed {len(foundedMemories) - len(deduplicatedMemories)} duplicate files.")
memories = deduplicatedMemories

Removing duplicates: 100%|██████████| 10568/10568 [23:07<00:00,  7.62it/s]

Removed 667 duplicate files.


_Remove too Short/Long Videos_

In [31]:
def getDuration(videoPath: str):
    try:
        cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", videoPath]
        result = subprocess.run(cmd, capture_output=True, text=True)
        return float(json.loads(result.stdout)["format"]["duration"])
    except Exception:
        return None

validMemories = []
removedCount = 0
for memory in tqdm(memories, desc="Filtering videos by duration"):
    if memory.endswith(".mp4"):
        duration = getDuration(memory)
        if duration is not None and (duration < 5 or duration > 150):
            removedCount += 1
            continue
        validMemories.append(memory)
    else:
        validMemories.append(memory)
memories = validMemories
print(f"Removed {removedCount} videos (duration < 5s or > 150s).")

_Get File Dates_

In [32]:
def getTakenDate(filePath):
    ext = os.path.splitext(filePath)[1].lower()

    if ext in ('.jpg', '.jpeg'):
        exifData = getattr(Image.open(filePath), '_getexif', lambda: None)()
        if exifData:
            for tag, value in exifData.items():
                if TAGS.get(tag) == 'DateTimeOriginal':
                    return datetime.strptime(value, '%Y:%m:%d %H:%M:%S').strftime('%B %Y')

    elif ext == '.png':
        img = Image.open(filePath)
        xmp = img.info.get('XML:com.adobe.xmp', '')
        if xmp:
            import re
            match = re.search(r'<exif:DateTimeOriginal>(.*?)</exif:DateTimeOriginal>', xmp)
            if match:
                try:
                    return datetime.fromisoformat(match.group(1)[:10]).strftime('%B %Y')
                except Exception:
                    pass

    elif ext == '.mp4':
        tags = MP4(filePath).tags
        if tags:
            for key in ('©day', 'date', 'DATE'):
                if key in tags:
                    try:
                        return datetime.fromisoformat(tags[key][0][:10]).strftime('%B %Y')
                    except Exception:
                        pass

        try:
            cmd = ["ffprobe", "-v", "error", "-show_entries", "format_tags=creation_time",
                   "-of", "json", filePath]
            result = subprocess.run(cmd, capture_output=True, text=True)
            creation_time = json.loads(result.stdout)["format"]["tags"]["creation_time"]
            return datetime.fromisoformat(creation_time[:10]).strftime('%B %Y')
        except Exception:
            pass

    return None

folderStructure = {
    "Photos": {},
    "Videos": {}
}

for memory in tqdm(memories, desc="Getting file dates"):
    date = getTakenDate(memory)
    goesTo = "Videos" if str(memory).lower().endswith(".mp4") else "Photos"

    if date is not None:
        month, year = date.split()

        if year not in folderStructure[goesTo]:
            folderStructure[goesTo][year] = {}
        if month not in folderStructure[goesTo][year]:
            folderStructure[goesTo][year][month] = []
        folderStructure[goesTo][year][month].append(memory)
    else:
        if "Unknown" not in folderStructure[goesTo]:
            folderStructure[goesTo]["Unknown"] = []
        folderStructure[goesTo]["Unknown"].append(memory)


_Organizing_

In [ ]:
outputRoot = os.path.join(os.getcwd(), "Memories")

copyTasks = []

for mediaType, years in folderStructure.items():
    for yearOrUnknown, value in years.items():
        if yearOrUnknown == "Unknown":
            destFolder = os.path.join(outputRoot, mediaType, "Unknown")
            os.makedirs(destFolder, exist_ok=True)
            for filePath in value:
                destFile = os.path.join(destFolder, os.path.basename(filePath))
                copyTasks.append((filePath, destFile))
        else:
            for month, files in value.items():
                destFolder = os.path.join(outputRoot, mediaType, yearOrUnknown, month)
                os.makedirs(destFolder, exist_ok=True)
                for filePath in files:
                    destFile = os.path.join(destFolder, os.path.basename(filePath))
                    copyTasks.append((filePath, destFile))

totalCopied = 0
totalSkipped = 0

for filePath, destFile in tqdm(copyTasks, desc="Copying files"):
    if not os.path.exists(destFile):
        shutil.copy2(filePath, destFile)
        totalCopied += 1
    else:
        totalSkipped += 1

print(f"Done. Copied: {totalCopied} files, Skipped (already exist): {totalSkipped} files.")
print(f"Output folder: {outputRoot}")

Videos/2024/July:  78%|███████▊  | 21/27 [00:12<00:03,  1.70it/s]


OSError: [Errno 28] No space left on device